# qwen-vlm — Colab 빠른 시작

이 노트북은 **데스크톱 실험 GUI 없이** CLI로 최소 실행을 검증하기 위한 것입니다.

- **저장소 URL**을 본인 GitHub 등으로 바꿔 클론하거나, zip 업로드 후 `%cd`로 이동하세요.
- **`LLAMA_LINUX_URL`**: [llama.cpp Releases](https://github.com/ggml-org/llama.cpp/releases)에서 **Linux + CUDA + x64** `.tar.gz` 직접 링크로 교체해야 합니다(릴리스마다 이름이 바뀜).
- 패키지는 **Python ≥ 3.12** 필요. 런타임 버전이 낮으면 Colab 이미지를 바꿉니다.
- 자세한 설명은 루트 `사용법-Colab.md`를 참고하세요.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 12), "Python 3.12 이상 런타임으로 바꿔 주세요."

## PyTorch (CUDA)

아래는 **예시**입니다. [pytorch.org](https://pytorch.org/get-started/locally/)에서 본인 Colab CUDA에 맞는 한 줄로 바꿀 수 있습니다.

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

## 저장소 받기 및 editable 설치
`REPO_URL`을 수정하거나, 업로드한 폴더로 `%cd`만 사용하세요.

In [ ]:
# REPO_URL = "https://github.com/<YOU>/<YOUR-REPO>.git"
# !git clone {REPO_URL} repo && %cd repo

In [ ]:
!pip install -U pip
!pip install -e .
!pip install ultralytics opencv-python-headless huggingface-hub psutil datasets matplotlib

## llama-server (Linux)
`LLAMA_LINUX_URL`을 릴리스 페이지의 **Linux CUDA** 아카이브 URL로 바꾼 뒤 실행하세요. 압축 안의 `llama-server` 경로는 릴리스마다 다를 수 있어 `find`로 확인합니다.

In [ ]:
import shutil
import tempfile
from pathlib import Path
from urllib.request import urlretrieve

# llama.cpp 릴리스에서 Linux + CUDA + x64 의 .tar.gz URL 로 바꾸세요.
LLAMA_LINUX_URL = "https://github.com/ggml-org/llama.cpp/releases/download/REPLACE/REPLACE-linux-cuda-x64.tar.gz"
VENDOR = Path("vendor/llama-cpp-linux-cuda")
VENDOR.mkdir(parents=True, exist_ok=True)

if "REPLACE" in LLAMA_LINUX_URL:
    raise RuntimeError("LLAMA_LINUX_URL을 llama.cpp 릴리스의 실제 Linux CUDA 다운로드 URL로 바꿔 주세요.")

blob = tempfile.gettempdir() + "/llama_linux.tgz"
urlretrieve(LLAMA_LINUX_URL, blob)
shutil.unpack_archive(blob, extract_dir=VENDOR, format="gztar")

cands = [p for p in VENDOR.rglob("llama-server") if p.is_file()]
print("Found:", cands)
assert cands, "llama-server 바이너리를 찾지 못했습니다. 압축 구조를 확인하세요."
LLAMA_SERVER = cands[0]
LLAMA_SERVER.chmod(0o755)
LLAMA_SERVER = str(LLAMA_SERVER.resolve())
print("Using:", LLAMA_SERVER)

## Qwen3-VL GGUF (Hugging Face)

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

out = Path("vendor/qwen3-vl-4b-q8-gguf")
out.mkdir(parents=True, exist_ok=True)
hf_hub_download("Qwen/Qwen3-VL-4B-Instruct-GGUF", "Qwen3VL-4B-Instruct-Q8_0.gguf", local_dir=str(out))
hf_hub_download("Qwen/Qwen3-VL-4B-Instruct-GGUF", "mmproj-Qwen3VL-4B-Instruct-Q8_0.gguf", local_dir=str(out))
print(list(out.iterdir()))

## HR-Bench 스모크 (샘플 2, `qwen_only`)
VRAM이 빠듯하면 `--max-samples 1` 로 줄이세요.

In [ ]:
import subprocess
from pathlib import Path

gguf = Path("vendor/qwen3-vl-4b-q8-gguf/Qwen3VL-4B-Instruct-Q8_0.gguf")
mmproj = Path("vendor/qwen3-vl-4b-q8-gguf/mmproj-Qwen3VL-4B-Instruct-Q8_0.gguf")
Path("docs").mkdir(exist_ok=True)

cmd = [
    "python",
    "-m",
    "qwen_vlm.cli.hr_bench",
    "--llama-server",
    LLAMA_SERVER,
    "--gguf",
    str(gguf),
    "--mmproj",
    str(mmproj),
    "--max-samples",
    "2",
    "--strategies",
    "qwen_only",
    "--json-out",
    "docs/hr_bench_last.json",
    "--html-out",
    "docs/hr_bench_last.html",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 결과 HTML 다운로드 (선택)

In [ ]:
from google.colab import files
files.download("docs/hr_bench_last.html")